In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

1.Cleaning the Raw Data

In [ ]:
df=pd.read_csv("cafe_sales.csv")
print(df.head())  # Display the first 5 rows

In [ ]:
print(df.isnull().sum())
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage.sort_values(ascending=False)

In [ ]:
df["Item"] = df["Item"].replace(["ERROR", "UNKNOWN"], np.nan)
df.dropna(subset=["Item"],inplace=True)

In [ ]:
df["Location"] = df["Location"].replace(["ERROR", "UNKNOWN"], np.nan)
df["Location"].fillna("In-store", inplace=True)

In [ ]:
payment_perc = df["Payment Method"].value_counts(normalize=True) * 100

print(payment_perc)

In [ ]:
df["Payment Method"] = df["Payment Method"].replace(["ERROR", "UNKNOWN"], np.nan)
df["Payment Method"].fillna("Cash", inplace=True)

In [ ]:
df["Quantity"]=df["Quantity"].replace(["ERROR","UNKNOWN"],np.nan)
df.dropna(subset=["Quantity"],inplace=True)

In [ ]:
df["Total Spent"]=df["Total Spent"].replace(["ERROR","UNKNOWN"],np.nan)
df["Price Per Unit"]=df["Price Per Unit"].replace(["ERROR","UNKNOWN"],np.nan)

In [ ]:
df["Quantity"] = pd.to_numeric(df["Quantity"])
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"])

In [ ]:
df["Total Spent"] = df["Quantity"] * df["Price Per Unit"]

In [ ]:
print(df[["Item", "Quantity", "Price Per Unit", "Total Spent"]].head())

In [ ]:
df["Price Per Unit"] = df["Price Per Unit"].replace(["ERROR", "UNKNOWN"], np.nan)
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")

In [ ]:
df["Price Per Unit"].fillna(1, inplace=True)

In [ ]:
df["Transaction Date"]=df["Transaction Date"].replace(["ERROR","UNKNOWN"], np.nan)


In [ ]:
df["Transaction Date"].fillna(pd.to_datetime("2023-03-31"), inplace=True)

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])
df["Transaction Date"] = df["Transaction Date"].dt.date

print(df["Transaction Date"].head())

In [ ]:
df.isnull().sum()

In [ ]:
df.to_csv("cafe_sales_cleaned.csv", index=False)

2. Adding Some KPI's

In [ ]:
# 1. AOV
aov = df["Total Spent"].mean()
total_revenue = df["Total Spent"].sum()

# 2. Volume vs Revenue
item_stats = df.groupby("Item").agg(
    Volume=("Quantity", "sum"), Revenue=("Total Spent", "sum")
)

# 3. Peak Days & Months
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])
df["Month"] = df["Transaction Date"].dt.to_period("M").astype(str)
df["Day_of_Week"] = df["Transaction Date"].dt.day_name()

monthly_rev = df.groupby("Month")["Total Spent"].sum()
daily_rev = df.groupby("Day_of_Week")["Total Spent"].sum()

print("=" * 45)
print("       CAFE SALES KPI SUMMARY REPORT       ")
print("=" * 45)
print(f"Total Revenue           : ${total_revenue:,.2f}")
print(f"Total Transactions      : {len(df):,}")
print(f"Average Order Value (AOV): ${aov:.2f}")
print("-" * 45)
print(
    f"Top Item (by Volume)   : {item_stats['Volume'].idxmax()} ({item_stats['Volume'].max():,} units)"
)
print(
    f"Top Item (by Revenue)  : {item_stats['Revenue'].idxmax()} (${item_stats['Revenue'].max():,.2f})"
)
print("-" * 45)
print(
    f"Peak Sales Month       : {monthly_rev.idxmax()} (${monthly_rev.max():,.2f})"
)
print(
    f"Peak Sales Day         : {daily_rev.idxmax()} (${daily_rev.max():,.2f})"
)
print("=" * 45)

3. Preparing Visualizations

In [ ]:
# [Visualization 1] Total Revenue by Item
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

item_summary = (
    df.groupby("Item")["Total Spent"].sum().sort_values(ascending=False)
)

ax = sns.barplot(
    x=item_summary.index, y=item_summary.values, palette="Blues_r"
)
plt.title("Total Revenue by Item ($)", fontsize=14, fontweight="bold")
plt.xlabel("Item", fontsize=12)
plt.ylabel("Revenue ($)", fontsize=12)

#
for p in ax.patches:
    ax.annotate(
        f"${p.get_height():,.0f}",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=10,
        xytext=(0, 3),
        textcoords="offset points",
    )

plt.tight_layout()
plt.savefig("chart_1_revenue_by_item.png", dpi=300)
plt.show()

In [ ]:
# [Visualization 2] Monthly Revenue Trend

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

df["Month"] = (
    pd.to_datetime(df["Transaction Date"]).dt.to_period("M").astype(str)
)
monthly = df.groupby("Month")["Total Spent"].sum().reset_index()

sns.lineplot(
    data=monthly,
    x="Month",
    y="Total Spent",
    marker="o",
    linewidth=2.5,
    color="#1f77b4",
)
plt.title("Monthly Revenue Trend (2023)", fontsize=14, fontweight="bold")
plt.xlabel("Month", fontsize=12)
plt.ylabel("Revenue ($)", fontsize=12)
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig("chart_2_monthly_trend.png", dpi=300)
plt.show()

In [ ]:
# [Visualization 3] Revenue by Location

sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 6))

loc_summary = df.groupby("Location")["Total Spent"].sum().reset_index()

ax = sns.barplot(
    data=loc_summary, x="Location", y="Total Spent", palette="Set2"
)
plt.title("Revenue by Location", fontsize=14, fontweight="bold")
plt.xlabel("Location", fontsize=12)
plt.ylabel("Revenue ($)", fontsize=12)

# Value annotations
for p in ax.patches:
    ax.annotate(
        f"${p.get_height():,.0f}",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=10,
        xytext=(0, 3),
        textcoords="offset points",
    )

plt.tight_layout()
plt.savefig("chart_3_revenue_by_location.png", dpi=300)
plt.show()

In [ ]:
# [Visualization 4] Payment Method Share

sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 8))

pay_summary = df["Payment Method"].value_counts()

plt.pie(
    pay_summary,
    labels=pay_summary.index,
    autopct="%1.1f%%",
    colors=sns.color_palette("pastel"),
    startangle=140,
)
plt.title("Payment Method Share", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("chart_4_payment_share.png", dpi=300)
plt.show()